In [1]:
# ============================================================
# FASE 4 — DATASET INTEGRITY
# Notebook: 01_dataset_verification.ipynb
# ============================================================

from google.colab import drive
from pathlib import Path
import os
import json
import hashlib
import shutil
import subprocess
import platform
from datetime import datetime

import numpy as np
import torch
import torchvision

# ------------------------------------------------------------
# Mount Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# Project locations
# ------------------------------------------------------------

PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/skripsi_label_noise"
)

REPO_DIR = Path(
    "/content/label-noise-thesis"
)

DATA_ROOT = Path(
    "/content/label_noise_data"
)

CIFAR_ROOT = DATA_ROOT / "cifar"

CIFARN_REPO = Path(
    "/content/cifar-n-official"
)

MANIFEST_DRIVE = (
    PROJECT_DRIVE / "manifests"
)

# ------------------------------------------------------------
# Create local directories
# ------------------------------------------------------------

DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CIFAR_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DRIVE.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_DRIVE :", PROJECT_DRIVE)
print("DATA_ROOT     :", DATA_ROOT)
print("CIFAR_ROOT    :", CIFAR_ROOT)
print("CIFARN_REPO   :", CIFARN_REPO)

Mounted at /content/drive
PROJECT_DRIVE : /content/drive/MyDrive/skripsi_label_noise
DATA_ROOT     : /content/label_noise_data
CIFAR_ROOT    : /content/label_noise_data/cifar
CIFARN_REPO   : /content/cifar-n-official


In [2]:
# ============================================================
# DOWNLOAD CIFAR-10
# ============================================================

from torchvision.datasets import CIFAR10

print("Downloading/loading CIFAR-10 training set...")

cifar10_train = CIFAR10(
    root=str(CIFAR_ROOT),
    train=True,
    download=True
)

print("Downloading/loading CIFAR-10 test set...")

cifar10_test = CIFAR10(
    root=str(CIFAR_ROOT),
    train=False,
    download=True
)

print()
print("=" * 70)
print("CIFAR-10 BASIC CHECK")
print("=" * 70)

print(
    "Training images :",
    len(cifar10_train)
)

print(
    "Test images     :",
    len(cifar10_test)
)

print(
    "Number classes  :",
    len(cifar10_train.classes)
)

print(
    "Classes         :",
    cifar10_train.classes
)

print(
    "Train data shape:",
    cifar10_train.data.shape
)

print(
    "Train dtype     :",
    cifar10_train.data.dtype
)

# ------------------------------------------------------------
# Hard assertions
# ------------------------------------------------------------

assert len(cifar10_train) == 50_000
assert len(cifar10_test) == 10_000

assert cifar10_train.data.shape == (
    50_000,
    32,
    32,
    3
)

assert cifar10_test.data.shape == (
    10_000,
    32,
    32,
    3
)

assert len(cifar10_train.classes) == 10

print()
print("✅ CIFAR-10 basic structure verified.")

Downloading/loading CIFAR-10 training set...


100%|██████████| 170M/170M [26:15<00:00, 108kB/s]


Downloading/loading CIFAR-10 test set...

CIFAR-10 BASIC CHECK
Training images : 50000
Test images     : 10000
Number classes  : 10
Classes         : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train data shape: (50000, 32, 32, 3)
Train dtype     : uint8

✅ CIFAR-10 basic structure verified.


In [3]:
# ============================================================
# VERIFY OFFICIAL CIFAR-10 ARCHIVE MD5
# ============================================================

import hashlib
from pathlib import Path

EXPECTED_CIFAR10_MD5 = (
    "c58f30108f718f92721af3b95e74349a"
)

archive_path = (
    CIFAR_ROOT / "cifar-10-python.tar.gz"
)

print(
    "Looking for archive:",
    archive_path
)

assert archive_path.exists(), (
    "CIFAR-10 archive tidak ditemukan."
)


def md5_file(
    path,
    chunk_size=1024 * 1024
):
    hasher = hashlib.md5()

    with open(path, "rb") as file:
        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            hasher.update(chunk)

    return hasher.hexdigest()


actual_md5 = md5_file(
    archive_path
)

print()
print("Expected MD5 :", EXPECTED_CIFAR10_MD5)
print("Actual MD5   :", actual_md5)

assert (
    actual_md5
    ==
    EXPECTED_CIFAR10_MD5
), "CIFAR-10 MD5 MISMATCH!"

print()
print("✅ CIFAR-10 official MD5 verified.")

Looking for archive: /content/label_noise_data/cifar/cifar-10-python.tar.gz

Expected MD5 : c58f30108f718f92721af3b95e74349a
Actual MD5   : c58f30108f718f92721af3b95e74349a

✅ CIFAR-10 official MD5 verified.


In [4]:
# ============================================================
# CLONE OFFICIAL CIFAR-N REPOSITORY
# ============================================================

import shutil
import subprocess

CIFARN_URL = (
    "https://github.com/UCSC-REAL/"
    "cifar-10-100n.git"
)

# Hapus hanya clone lokal runtime bila sudah ada
if CIFARN_REPO.exists():
    shutil.rmtree(
        CIFARN_REPO
    )

result = subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        CIFARN_URL,
        str(CIFARN_REPO)
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

assert CIFARN_REPO.exists(), (
    "Official CIFAR-N repository gagal di-clone."
)

# ------------------------------------------------------------
# Capture commit
# ------------------------------------------------------------

commit_sha = subprocess.run(
    [
        "git",
        "-C",
        str(CIFARN_REPO),
        "rev-parse",
        "HEAD"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print()
print("=" * 70)
print("OFFICIAL CIFAR-N")
print("=" * 70)

print(
    "Repository:",
    CIFARN_URL
)

print(
    "Commit SHA:",
    commit_sha
)


Cloning into '/content/cifar-n-official'...


OFFICIAL CIFAR-N
Repository: https://github.com/UCSC-REAL/cifar-10-100n.git
Commit SHA: 49df7d8a69e355470c77c1c2f2424916325a394b


In [5]:
# ============================================================
# CHECK CIFAR-N REQUIRED FILES
# ============================================================

CIFAR10N_FILE = (
    CIFARN_REPO
    / "data"
    / "CIFAR-10_human.pt"
)

IMAGE_ORDER_FILE = (
    CIFARN_REPO
    / "image_order_c10.npy"
)

required_files = [
    CIFAR10N_FILE,
    IMAGE_ORDER_FILE,
]

print("=" * 70)
print("CIFAR-N FILE CHECK")
print("=" * 70)

for file in required_files:

    exists = file.exists()

    size_mb = (
        file.stat().st_size / 1024**2
        if exists
        else 0
    )

    print(
        f"{file.name:<30}",
        f"exists={exists}",
        f"size={size_mb:.3f} MB"
    )

    assert exists, (
        f"File CIFAR-N tidak ditemukan: {file}"
    )

print()
print("✅ Required CIFAR-N files available.")

CIFAR-N FILE CHECK
CIFAR-10_human.pt              exists=True size=2.290 MB
image_order_c10.npy            exists=True size=0.382 MB

✅ Required CIFAR-N files available.


In [6]:
# ============================================================
# SHA-256 MANIFEST CIFAR-N
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:

        while True:

            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            hasher.update(
                chunk
            )

    return hasher.hexdigest()


cifar10n_sha256 = sha256_file(
    CIFAR10N_FILE
)

image_order_sha256 = sha256_file(
    IMAGE_ORDER_FILE
)

print("=" * 70)
print("CIFAR-N SHA-256")
print("=" * 70)

print(
    "CIFAR-10_human.pt :",
    cifar10n_sha256
)

print(
    "image_order_c10   :",
    image_order_sha256
)

CIFAR-N SHA-256
CIFAR-10_human.pt : 873e69c39cb9b5e97fb6bae2d60bb59b38a5cbc31d2b868f7903dcb2b9dd2310
image_order_c10   : db41bd6c919c154739e796d50ea811feffc77a8ab13a72639b9806cf708f1724


In [7]:
# ============================================================
# LOAD CIFAR-10N HUMAN LABELS
# ============================================================

cifar10n = torch.load(
    CIFAR10N_FILE,
    map_location="cpu",
    weights_only=False
)

print("=" * 70)
print("CIFAR-10N CONTENT")
print("=" * 70)

print(
    "Type:",
    type(cifar10n)
)

print(
    "Keys:",
    list(cifar10n.keys())
)

CIFAR-10N CONTENT
Type: <class 'dict'>
Keys: ['clean_label', 'aggre_label', 'worse_label', 'random_label1', 'random_label2', 'random_label3']


In [8]:
# ============================================================
# CIFAR-10N KEY VALIDATION
# ============================================================

required_keys = {
    "clean_label",
    "worse_label",
    "aggre_label",
    "random_label1",
    "random_label2",
    "random_label3",
}

available_keys = set(
    cifar10n.keys()
)

missing_keys = (
    required_keys
    -
    available_keys
)

print(
    "Required keys:",
    sorted(required_keys)
)

print(
    "Available keys:",
    sorted(available_keys)
)

assert len(missing_keys) == 0, (
    f"Missing CIFAR-10N keys: {missing_keys}"
)

for key in required_keys:

    assert len(
        cifar10n[key]
    ) == 50_000, (
        f"{key} tidak memiliki 50.000 labels"
    )

print()
print(
    "✅ CIFAR-10N keys and lengths verified."
)

Required keys: ['aggre_label', 'clean_label', 'random_label1', 'random_label2', 'random_label3', 'worse_label']
Available keys: ['aggre_label', 'clean_label', 'random_label1', 'random_label2', 'random_label3', 'worse_label']

✅ CIFAR-10N keys and lengths verified.


In [9]:
# ============================================================
# CRITICAL SEMANTIC ALIGNMENT CHECK
# ============================================================

torchvision_clean_labels = np.asarray(
    cifar10_train.targets,
    dtype=np.int64
)

cifarn_clean_labels = np.asarray(
    cifar10n["clean_label"],
    dtype=np.int64
)

print("=" * 70)
print("CRITICAL IMAGE/LABEL ALIGNMENT CHECK")
print("=" * 70)

print(
    "Torchvision labels shape:",
    torchvision_clean_labels.shape
)

print(
    "CIFAR-N labels shape    :",
    cifarn_clean_labels.shape
)

alignment_matches = (
    torchvision_clean_labels
    ==
    cifarn_clean_labels
)

n_matches = int(
    alignment_matches.sum()
)

n_mismatch = int(
    (~alignment_matches).sum()
)

print()
print(
    "Matching labels :",
    n_matches
)

print(
    "Mismatches      :",
    n_mismatch
)

assert np.array_equal(
    torchvision_clean_labels,
    cifarn_clean_labels
), (
    "FATAL: CIFAR-10 / CIFAR-10N IMAGE ORDER MISMATCH!"
)

print()
print("=" * 70)
print(
    "✅ CIFAR-10 ↔ CIFAR-10N ALIGNMENT VERIFIED"
)
print("=" * 70)

CRITICAL IMAGE/LABEL ALIGNMENT CHECK
Torchvision labels shape: (50000,)
CIFAR-N labels shape    : (50000,)

Matching labels : 50000
Mismatches      : 0

✅ CIFAR-10 ↔ CIFAR-10N ALIGNMENT VERIFIED


In [10]:
# ============================================================
# COMPUTE CIFAR-10N HUMAN NOISE RATES
# ============================================================

def calculate_noise_rate(
    clean_labels,
    noisy_labels
):
    clean_labels = np.asarray(
        clean_labels
    )

    noisy_labels = np.asarray(
        noisy_labels
    )

    assert (
        clean_labels.shape
        ==
        noisy_labels.shape
    )

    error_mask = (
        clean_labels
        !=
        noisy_labels
    )

    return float(
        error_mask.mean()
    )


noise_keys = [
    "aggre_label",
    "random_label1",
    "random_label2",
    "random_label3",
    "worse_label",
]

noise_rates = {}

print("=" * 70)
print("CIFAR-10N HUMAN NOISE RATES")
print("=" * 70)

for key in noise_keys:

    rate = calculate_noise_rate(
        cifar10n["clean_label"],
        cifar10n[key]
    )

    noise_rates[key] = rate

    print(
        f"{key:<18}: "
        f"{100 * rate:.2f}%"
    )

CIFAR-10N HUMAN NOISE RATES
aggre_label       : 9.01%
random_label1     : 17.23%
random_label2     : 18.12%
random_label3     : 17.64%
worse_label       : 40.21%


In [11]:
# ============================================================
# HUMAN NOISE SANITY CHECK
# ============================================================

assert (
    0.08
    <
    noise_rates["aggre_label"]
    <
    0.10
), "Aggregate noise rate di luar expected range."

assert (
    0.38
    <
    noise_rates["worse_label"]
    <
    0.42
), "Worst noise rate di luar expected range."

for key in [
    "random_label1",
    "random_label2",
    "random_label3",
]:
    assert (
        0.15
        <
        noise_rates[key]
        <
        0.20
    ), (
        f"{key} noise rate di luar expected range."
    )

print()
print(
    "✅ CIFAR-10N human-noise rates passed sanity checks."
)


✅ CIFAR-10N human-noise rates passed sanity checks.


In [12]:
# ============================================================
# CLASS DISTRIBUTION CHECK
# ============================================================

unique_classes, counts = np.unique(
    torchvision_clean_labels,
    return_counts=True
)

print("=" * 70)
print("CIFAR-10 TRAIN CLASS DISTRIBUTION")
print("=" * 70)

for class_id, count in zip(
    unique_classes,
    counts
):

    class_name = (
        cifar10_train.classes[
            int(class_id)
        ]
    )

    print(
        f"{class_id:>2} "
        f"{class_name:<12} "
        f"{count}"
    )

assert len(
    unique_classes
) == 10

assert np.all(
    counts == 5000
)

print()
print(
    "✅ Training class distribution verified."
)

CIFAR-10 TRAIN CLASS DISTRIBUTION
 0 airplane     5000
 1 automobile   5000
 2 bird         5000
 3 cat          5000
 4 deer         5000
 5 dog          5000
 6 frog         5000
 7 horse        5000
 8 ship         5000
 9 truck        5000

✅ Training class distribution verified.


In [13]:
# ============================================================
# CREATE DATASET MANIFEST
# ============================================================

from datetime import datetime
import json

dataset_manifest = {

    "created_at":
        datetime.now().isoformat(),

    "phase":
        "PHASE_4_DATASET_INTEGRITY",

    # --------------------------------------------------------
    # CIFAR-10
    # --------------------------------------------------------

    "cifar10": {

        "source":
            "University of Toronto / torchvision",

        "version":
            "Python",

        "train_images":
            len(cifar10_train),

        "test_images":
            len(cifar10_test),

        "image_shape":
            list(
                cifar10_train.data.shape[1:]
            ),

        "num_classes":
            len(
                cifar10_train.classes
            ),

        "classes":
            list(
                cifar10_train.classes
            ),

        "archive_md5":
            actual_md5,

        "expected_official_md5":
            EXPECTED_CIFAR10_MD5,

        "md5_verified":
            (
                actual_md5
                ==
                EXPECTED_CIFAR10_MD5
            ),
    },

    # --------------------------------------------------------
    # CIFAR-10N
    # --------------------------------------------------------

    "cifar10n": {

        "official_repository":
            CIFARN_URL,

        "git_commit":
            commit_sha,

        "human_label_file":
            "data/CIFAR-10_human.pt",

        "human_label_sha256":
            cifar10n_sha256,

        "image_order_file":
            "image_order_c10.npy",

        "image_order_sha256":
            image_order_sha256,

        "available_keys":
            sorted(
                list(
                    cifar10n.keys()
                )
            ),

        "noise_rates": {
            key: float(value)
            for key, value
            in noise_rates.items()
        },
    },

    # --------------------------------------------------------
    # Alignment
    # --------------------------------------------------------

    "alignment": {

        "torchvision_vs_cifarn_clean_label":
            True,

        "matching_samples":
            n_matches,

        "mismatches":
            n_mismatch,

        "mapping_applied":
            False,

        "reason":
            (
                "Using torchvision Python CIFAR and "
                "PyTorch CIFAR-N .pt labels; direct "
                "clean-label alignment assertion passed."
            ),
    },

    # --------------------------------------------------------
    # Environment
    # --------------------------------------------------------

    "environment": {

        "python":
            platform.python_version(),

        "torch":
            torch.__version__,

        "torchvision":
            torchvision.__version__,
    },

    "status":
        "VERIFIED"
}


# ------------------------------------------------------------
# Save to Drive
# ------------------------------------------------------------

manifest_path = (
    MANIFEST_DRIVE
    / "dataset_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        dataset_manifest,
        file,
        indent=4
    )


print("=" * 70)
print("DATASET MANIFEST SAVED")
print("=" * 70)

print(
    manifest_path
)

DATASET MANIFEST SAVED
/content/drive/MyDrive/skripsi_label_noise/manifests/dataset_manifest.json


In [14]:
# ============================================================
# FINAL PHASE-4 VERIFICATION REPORT
# ============================================================

report = {

    "cifar10_train_50000":
        len(cifar10_train) == 50_000,

    "cifar10_test_10000":
        len(cifar10_test) == 10_000,

    "cifar10_shape_valid":
        cifar10_train.data.shape
        ==
        (50_000, 32, 32, 3),

    "cifar10_md5_valid":
        actual_md5
        ==
        EXPECTED_CIFAR10_MD5,

    "cifar10n_file_exists":
        CIFAR10N_FILE.exists(),

    "cifar10n_keys_valid":
        len(missing_keys) == 0,

    "alignment_valid":
        n_mismatch == 0,

    "aggregate_noise_valid":
        (
            0.08
            <
            noise_rates["aggre_label"]
            <
            0.10
        ),

    "worst_noise_valid":
        (
            0.38
            <
            noise_rates["worse_label"]
            <
            0.42
        ),
}


print("=" * 70)
print("FASE 4 — DATASET INTEGRITY REPORT")
print("=" * 70)

all_passed = True

for check, passed in report.items():

    symbol = (
        "✅"
        if passed
        else "❌"
    )

    print(
        f"{symbol} {check}"
    )

    if not passed:
        all_passed = False


print()

if all_passed:

    print("=" * 70)
    print(
        "FASE 4 CORE DATASET VERIFICATION PASSED"
    )
    print("=" * 70)

else:

    raise RuntimeError(
        "FASE 4 verification FAILED. "
        "Jangan lanjut ke synthetic noise."
    )

FASE 4 — DATASET INTEGRITY REPORT
✅ cifar10_train_50000
✅ cifar10_test_10000
✅ cifar10_shape_valid
✅ cifar10_md5_valid
✅ cifar10n_file_exists
✅ cifar10n_keys_valid
✅ alignment_valid
✅ aggregate_noise_valid
✅ worst_noise_valid

FASE 4 CORE DATASET VERIFICATION PASSED
